<a href="https://colab.research.google.com/github/Carolaynebarret/DataConnect/blob/main/Mini_Desafio_DataConnec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Carregamento dos Dados e Função de Diagnóstico

Primeiro, vamos importar as bibliotecas necessárias, definir os caminhos dos arquivos e criar uma função auxiliar para adivinhar a chave primária de um DataFrame, que será usada para verificar duplicatas por chave. Em seguida, definimos a função principal `diagnose_dataframe` que executa todas as verificações solicitadas para um único DataFrame.

In [6]:
import pandas as pd
import numpy as np

# Caminhos dos arquivos CSV (assumindo que estão na pasta /content/dados/)
file_paths = {
    'apontamentos': '/content/dados/dc_apontamentos.csv',
    'projetos': '/content/dados/dc_projetos.csv',
    'clientes': '/content/dados/dc_clientes.csv',
    'analistas': '/content/dados/dc_analistas.csv',
    'satisfacao': '/content/dados/dc_satisfacao.csv',
}

# Dicionário para armazenar os DataFrames
dfs = {}

In [7]:
# Função auxiliar para tentar adivinhar a chave primária de um DataFrame
def guess_primary_key(df_name, df_columns):
    potential_keys = [
        f'{df_name}_id',
        'id',
        f'cd_{df_name.replace('dc_', '')}', # Adaptado para 'dc_projetos' -> 'cd_projetos'
        f'{df_name.replace('dc_', '')}Id',
        'ID'
    ]
    for pk in potential_keys:
        if pk in df_columns:
            return pk
    return None

In [8]:
def diagnose_dataframe(df_name, df):
    print(f"\n{'='*50}\n--- Diagnóstico para: {df_name.upper()} ---\n{'='*50}")

    # 1. Quantas linhas e colunas
    print(f"\n1. Dimensões do DataFrame: {df.shape[0]} linhas, {df.shape[1]} colunas")

    # 2. Nomes das colunas
    print("\n2. Nomes das Colunas:")
    for col in df.columns:
        print(f"  - {col}")

    # 3. 10 primeiras linhas
    print("\n3. 10 Primeiras Linhas:")
    display(df.head(10))

    # 4. Tipos de cada coluna
    print("\n4. Tipos de Dados das Colunas (`df.info()`):")
    df.info()

    # 5. Nulos por coluna
    null_counts = df.isnull().sum()
    print("\n5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):")
    if null_counts.sum() > 0:
        display(null_counts[null_counts > 0])
    else:
        print("  Não há valores nulos em nenhuma coluna.")

    # 6. Linhas duplicadas
    full_duplicates = df.duplicated().sum()
    print(f"\n6. Linhas Duplicadas Completas: {full_duplicates}")

    # Duplicatas pela chave primária
    pk = guess_primary_key(df_name, df.columns)
    if pk:
        pk_duplicates = df.duplicated(subset=[pk]).sum()
        print(f"   Duplicatas pela chave primária '{pk}': {pk_duplicates}")
        if pk_duplicates > 0:
            print(f"   Exemplos de linhas duplicadas pela chave '{pk}':")
            display(df[df.duplicated(subset=[pk], keep=False)].sort_values(by=pk).head())
    else:
        print(f"   AVISO: Não foi possível identificar uma chave primária (ex: 'id' ou '{df_name}_id') para '{df_name}'. Não foi possível verificar duplicatas por chave.")

    # 7. Valores únicos de colunas categóricas e espaços em branco
    print("\n7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):")
    object_cols = df.select_dtypes(include='object').columns
    if object_cols.empty:
        print("  Não há colunas do tipo 'object' para analisar.")
    else:
        for col in object_cols:
            print(f"\n  - Coluna '{col}':")
            unique_vals = df[col].astype(str).unique()
            if len(unique_vals) <= 50: # Mostrar valores únicos se não forem muitos
                print(f"    Valores únicos ({len(unique_vals)}): {unique_vals.tolist()}")
            else:
                print(f"    {len(unique_vals)} valores únicos (mostrando os 5 primeiros): {unique_vals[:5].tolist()}...")

            # Verificar espaços em branco no início/fim
            # Usar str.strip() para garantir que espaços em branco sejam identificados
            has_leading_trailing_spaces = df[col].astype(str).apply(lambda x: x != x.strip()).any()
            if has_leading_trailing_spaces:
                # Encontrar um exemplo de valor com espaços
                example_value = df[col][df[col].astype(str).apply(lambda x: x != x.strip())].iloc[0]
                print(f"    AVISO: Existem valores com espaços em branco no início ou fim nesta coluna. Exemplo: '{example_value}'")
            else:
                print("    Não foram encontrados valores com espaços em branco no início ou fim.")

    # 8. Datas: formatos inconsistentes
    print("\n8. Análise de Colunas de Data:")
    # Incluir colunas object e datetime64 para análise
    potential_date_cols = df.select_dtypes(include=['object', 'datetime64']).columns.tolist()
    found_inconsistent_date = False

    if not potential_date_cols:
        print("  Nenhuma coluna de data potencial identificada.")
    else:
        for col in potential_date_cols:
            # Tentar converter para datetime, forçando erros para NaT
            temp_date_series = pd.to_datetime(df[col], errors='coerce')

            # Se houver NaTs introduzidos, mas não todas as linhas, há inconsistência ou valores não-data
            num_na_after_coerce = temp_date_series.isnull().sum()
            num_na_original = df[col].isnull().sum()

            if df[col].dtype == 'object' and num_na_after_coerce > num_na_original:
                # Se mais NaNs foram introduzidos, significa que havia valores que não eram datas válidas
                print(f"  AVISO: Coluna '{col}' (tipo object) contém datas com formatos inconsistentes ou valores não-data.\n    Valores que não puderam ser convertidos: {df[col][temp_date_series.isnull() & ~df[col].isnull()].unique().tolist()}")
                found_inconsistent_date = True
            elif df[col].dtype == 'object' and num_na_after_coerce == 0 and num_na_original == 0:
                print(f"  Coluna '{col}' (tipo object) parece conter datas válidas em formato consistente (todos convertidos com sucesso).")
            elif df[col].dtype == 'datetime64':
                print(f"  Coluna '{col}' já é do tipo datetime64.")

        if not found_inconsistent_date:
            print("  Todas as colunas de data identificadas parecem ter formatos consistentes ou já são do tipo datetime.")

    # 9. Outliers
    print("\n9. Análise de Outliers (Exemplos Específicos e Estatísticas Descritivas):")
    numeric_cols = df.select_dtypes(include=np.number).columns
    if numeric_cols.empty:
        print("  Não há colunas numéricas para análise de outliers.")
    else:
        print("  Estatísticas descritivas para colunas numéricas (incluindo quartis para detecção simples de outliers):")
        display(df[numeric_cols].describe())

        # Verificações específicas de outliers
        if df_name == 'apontamentos' and 'horas_apontadas' in df.columns:
            outliers_apontamentos = df[df['horas_apontadas'] > 24]
            if not outliers_apontamentos.empty:
                print(f"\n  AVISO: '{df_name}' - Existem {len(outliers_apontamentos)} apontamentos com mais de 24 horas (`horas_apontadas` > 24):")
                display(outliers_apontamentos.head())
            else:
                print(f"\n  '{df_name}' - Nenhum apontamento com mais de 24 horas encontrado em `horas_apontadas`.")

        if df_name == 'projetos' and 'valor_contrato' in df.columns:
            # Detecção de outliers usando IQR para 'valor_contrato'
            Q1 = df['valor_contrato'].quantile(0.25)
            Q3 = df['valor_contrato'].quantile(0.75)
            IQR = Q3 - Q1
            upper_bound = Q3 + 1.5 * IQR
            lower_bound = Q1 - 1.5 * IQR
            outliers_contrato = df[(df['valor_contrato'] < lower_bound) | (df['valor_contrato'] > upper_bound)]
            if not outliers_contrato.empty:
                print(f"\n  AVISO: '{df_name}' - Existem {len(outliers_contrato)} projetos com `valor_contrato` fora dos limites IQR (possíveis outliers):")
                display(outliers_contrato.head())
            else:
                print(f"\n  '{df_name}' - Nenhuma anomalia de `valor_contrato` detectada pelo método IQR simples.")

    print(f"\n{'='*50}\n--- Fim do Diagnóstico para: {df_name.upper()} ---\n{'='*50}\n")

## 2. Executando o Diagnóstico para Cada Arquivo

Agora, vamos carregar cada um dos 5 arquivos CSV para DataFrames e aplicar a função `diagnose_dataframe` que acabamos de definir. Isso nos dará uma visão detalhada do estado de cada conjunto de dados individualmente.

In [9]:
# Carregar cada arquivo e executar o diagnóstico
for name, path in file_paths.items():
    try:
        dfs[name] = pd.read_csv(path)
        diagnose_dataframe(name, dfs[name])
    except FileNotFoundError:
        print(f"ERRO: Arquivo '{path}' não encontrado. Verifique o caminho ou se ele foi carregado corretamente.\n")
    except Exception as e:
        print(f"ERRO ao processar o arquivo '{path}': {e}\n")


--- Diagnóstico para: APONTAMENTOS ---

1. Dimensões do DataFrame: 9879 linhas, 6 colunas

2. Nomes das Colunas:
  - apontamento_id
  - projeto_id
  - analista_id
  - data
  - horas
  - atividade

3. 10 Primeiras Linhas:


,apontamento_id,projeto_id,analista_id,data,horas,atividade
0,APT002473,PRJ0032,ANL006,2026-03-23,6,Reunião com cliente
1,APT004798,PRJ0064,ANL008,2026-05-08,6.5,Apresentação
2,APT005566,PRJ0079,ANL012,2025-06-02,6,Coleta
3,APT000099,PRJ0003,ANL009,2025-09-15,4.5,Documentação
4,APT009367,PRJ0134,ANL009,2026-01-22,4,Modelagem
5,APT008582,PRJ0121,ANL009,05/04/2026,8,Testes
6,APT006641,PRJ0093,ANL020,11/09/2024,2,Testes
7,APT006423,PRJ0089,ANL022,2024-09-11,4.5,Coleta
8,APT007648,PRJ0109,ANL019,2026-03-27,6,Modelagem
9,APT006902,PRJ0098,ANL018,10/15/2025,7,Apresentação



4. Tipos de Dados das Colunas (`df.info()`):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9879 entries, 0 to 9878
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   apontamento_id  9879 non-null   object
 1   projeto_id      9879 non-null   object
 2   analista_id     9879 non-null   object
 3   data            9879 non-null   object
 4   horas           9879 non-null   object
 5   atividade       9879 non-null   object
dtypes: object(6)
memory usage: 463.2+ KB

5. Contagem de Valores Nulos por Coluna (apenas colunas com nulos):
  Não há valores nulos em nenhuma coluna.

6. Linhas Duplicadas Completas: 80
   AVISO: Não foi possível identificar uma chave primária (ex: 'id' ou 'apontamentos_id') para 'apontamentos'. Não foi possível verificar duplicatas por chave.

7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):

  - Coluna 'apontamento_id':
    9799 valores únicos (mostrando os 5 prim

/tmp/ipykernel_948/2298784008.py:78: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')


  AVISO: Coluna 'apontamento_id' (tipo object) contém datas com formatos inconsistentes ou valores não-data.
    Valores que não puderam ser convertidos: ['APT002473', 'APT004798', 'APT005566', 'APT000099', 'APT009367', 'APT008582', 'APT006641', 'APT006423', 'APT007648', 'APT006902', 'APT006332', 'APT000467', 'APT000326', 'APT006821', 'APT006338', 'APT000052', 'APT006146', 'APT009012', 'APT004016', 'APT002946', 'APT002753', 'APT000356', 'APT001350', 'APT004181', 'APT005769', 'APT001239', 'APT008005', 'APT003035', 'APT001870', 'APT000795', 'APT009722', 'APT005855', 'APT003487', 'APT006342', 'APT000298', 'APT009699', 'APT005766', 'APT003696', 'APT006649', 'APT006740', 'APT004965', 'APT000841', 'APT004107', 'APT003896', 'APT009276', 'APT003427', 'APT007556', 'APT007431', 'APT005657', 'APT007090', 'APT003422', 'APT002847', 'APT006915', 'APT002704', 'APT005319', 'APT007094', 'APT007116', 'APT006893', 'APT002495', 'APT008460', 'APT000536', 'APT003912', 'APT008128', 'APT006732', 'APT007581', 

...

,0
nota_nps,8



6. Linhas Duplicadas Completas: 0
   AVISO: Não foi possível identificar uma chave primária (ex: 'id' ou 'satisfacao_id') para 'satisfacao'. Não foi possível verificar duplicatas por chave.

7. Análise de Colunas Categóricas (Valores Únicos e Espaços em Branco):

  - Coluna 'pesquisa_id':
    103 valores únicos (mostrando os 5 primeiros): ['PSQ0001', 'PSQ0002', 'PSQ0003', 'PSQ0004', 'PSQ0005']...
    Não foram encontrados valores com espaços em branco no início ou fim.

  - Coluna 'projeto_id':
    103 valores únicos (mostrando os 5 primeiros): ['PRJ0001', 'PRJ0003', 'PRJ0004', 'PRJ0005', 'PRJ0006']...
    Não foram encontrados valores com espaços em branco no início ou fim.

  - Coluna 'data_pesquisa':
    100 valores únicos (mostrando os 5 primeiros): ['2024-05-27', '2025-10-20', '2024-05-08', '2025-03-27', '17/07/2025']...
    Não foram encontrados valores com espaços em branco no início ou fim.

  - Coluna 'comentario':
    Valores únicos (10): ['Resultado ok, prazo apertado.', 'D

/tmp/ipykernel_948/2298784008.py:78: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_948/2298784008.py:78: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')
/tmp/ipykernel_948/2298784008.py:78: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp_date_series = pd.to_datetime(df[col], errors='coerce')


,nota_nps
count,95.000000
mean,8.421053
std,1.601721
min,3.000000
25%,8.000000
50%,9.000000
75%,10.000000
max,10.000000



--- Fim do Diagnóstico para: SATISFACAO ---



## 3. Verificações de Consistência Entre Arquivos

Por fim, vamos realizar as verificações de consistência entre os DataFrames, garantindo que as chaves estrangeiras referenciem IDs existentes nos DataFrames relacionados.

In [10]:
print(f"\n{'='*50}\n--- Verificações de Consistência Entre Arquivos ---\n{'='*50}")

# 1. Verificar se todo projeto_id que aparece em apontamentos existe em projetos
if 'apontamentos' in dfs and 'projetos' in dfs and 'projeto_id' in dfs['apontamentos'].columns and 'projeto_id' in dfs['projetos'].columns:
    apontamentos_projetos_ids = dfs['apontamentos']['projeto_id'].dropna().unique()
    projetos_ids = dfs['projetos']['projeto_id'].dropna().unique()
    missing_projetos_in_apontamentos = np.setdiff1d(apontamentos_projetos_ids, projetos_ids)
    if len(missing_projetos_in_apontamentos) > 0:
        print(f"\nAVISO: 'projeto_id' em `apontamentos` não encontrados em `projetos`: {len(missing_projetos_in_apontamentos)} IDs.\n  Exemplos: {missing_projetos_in_apontamentos[:5].tolist()}")
    else:
        print("\nTodos os 'projeto_id' em `apontamentos` existem em `projetos`.")
else:
    print("\nNão foi possível verificar a consistência de 'projeto_id' entre `apontamentos` e `projetos` (arquivos/colunas ausentes ou renomeadas).")

# 2. Verificar se todo cliente_id de projetos existe em clientes
if 'projetos' in dfs and 'clientes' in dfs and 'cliente_id' in dfs['projetos'].columns and 'cliente_id' in dfs['clientes'].columns:
    projetos_clientes_ids = dfs['projetos']['cliente_id'].dropna().unique()
    clientes_ids = dfs['clientes']['cliente_id'].dropna().unique()
    missing_clientes_in_projetos = np.setdiff1d(projetos_clientes_ids, clientes_ids)
    if len(missing_clientes_in_projetos) > 0:
        print(f"\nAVISO: 'cliente_id' em `projetos` não encontrados em `clientes`: {len(missing_clientes_in_projetos)} IDs.\n  Exemplos: {missing_clientes_in_projetos[:5].tolist()}")
    else:
        print("\nTodos os 'cliente_id' em `projetos` existem em `clientes`.")
else:
    print("\nNão foi possível verificar a consistência de 'cliente_id' entre `projetos` e `clientes` (arquivos/colunas ausentes ou renomeadas).")

# 3. Verificar se todo analista_id de apontamentos existe em analistas
if 'apontamentos' in dfs and 'analistas' in dfs and 'analista_id' in dfs['apontamentos'].columns and 'analista_id' in dfs['analistas'].columns:
    apontamentos_analistas_ids = dfs['apontamentos']['analista_id'].dropna().unique()
    analistas_ids = dfs['analistas']['analista_id'].dropna().unique()
    missing_analistas_in_apontamentos = np.setdiff1d(apontamentos_analistas_ids, analistas_ids)
    if len(missing_analistas_in_apontamentos) > 0:
        print(f"\nAVISO: 'analista_id' em `apontamentos` não encontrados em `analistas`: {len(missing_analistas_in_apontamentos)} IDs.\n  Exemplos: {missing_analistas_in_apontamentos[:5].tolist()}")
    else:
        print("\nTodos os 'analista_id' em `apontamentos` existem em `analistas`.")
else:
    print("\nNão foi possível verificar a consistência de 'analista_id' entre `apontamentos` e `analistas` (arquivos/colunas ausentes ou renomeadas).")

# 4. Verificar se todo projeto_id de satisfacao existe em projetos
if 'satisfacao' in dfs and 'projetos' in dfs and 'projeto_id' in dfs['satisfacao'].columns and 'projeto_id' in dfs['projetos'].columns:
    satisfacao_projetos_ids = dfs['satisfacao']['projeto_id'].dropna().unique()
    projetos_ids = dfs['projetos']['projeto_id'].dropna().unique()
    missing_projetos_in_satisfacao = np.setdiff1d(satisfacao_projetos_ids, projetos_ids)
    if len(missing_projetos_in_satisfacao) > 0:
        print(f"\nAVISO: 'projeto_id' em `satisfacao` não encontrados em `projetos`: {len(missing_projetos_in_satisfacao)} IDs.\n  Exemplos: {missing_projetos_in_satisfacao[:5].tolist()}")
    else:
        print("\nTodos os 'projeto_id' em `satisfacao` existem em `projetos`.")
else:
    print("\nNão foi possível verificar a consistência de 'projeto_id' entre `satisfacao` e `projetos` (arquivos/colunas ausentes ou renomeadas).")

print(f"\n{'='*50}\n--- Fim das Verificações de Consistência ---\n{'='*50}\n")


--- Verificações de Consistência Entre Arquivos ---

Todos os 'projeto_id' em `apontamentos` existem em `projetos`.

Todos os 'cliente_id' em `projetos` existem em `clientes`.

Todos os 'analista_id' em `apontamentos` existem em `analistas`.

Todos os 'projeto_id' em `satisfacao` existem em `projetos`.

--- Fim das Verificações de Consistência ---

